# Aula 10 — Agentic RAG e Adaptive RAG
## Agentes de Recuperação com LangGraph

**MBA em RAG & CAG Aplicados a Direito e Segurança Pública**  
**Proporção: 20% Teoria / 80% Prática**  
**Duração estimada: 5 horas**

---

### Normas ABNT — Referências desta Aula

- YAO, Shunyu et al. **ReAct: Synergizing Reasoning and Acting in Language Models**. ICLR, 2023.
- JEONG, Soyeong et al. **Adaptive-RAG: Learning to Adapt Retrieval-Augmented Large Language Models through Question Complexity**. ACL Findings, 2024.
- LANGCHAIN-AI. **LangGraph Documentation**. Disponível em: https://langchain-ai.github.io/langgraph. Acesso em: 2024.

---

### Ementa

Implementação de sistemas RAG baseados em agentes que planejam e executam múltiplas chamadas de retrieval. Adaptive RAG com classificação dinâmica de estratégia baseada na complexidade da query. Aplicações no contexto jurídico e de segurança pública.

### Objetivos de Aprendizagem

Ao final desta aula, o aluno será capaz de:

1. Compreender o padrão ReAct e como ele possibilita raciocínio multi-etapa com retrieval
2. Implementar um agente RAG com 3 ferramentas distintas no LangGraph
3. Construir um classificador de complexidade para Adaptive RAG com 3 caminhos
4. Interpretar traces de agentes no LangFuse e identificar ineficiências
5. Avaliar agentes com RAGAS e DeepEval considerando a trajetória de execução

### Stack Tecnológico

`LangGraph` · `LangChain Agents` · `ReAct` · `Ollama` · `Tavily` · `SQLite` · `LangFuse` · `RAGAS` · `DeepEval`

---
## Parte 1 — Fundamentos Teóricos (20%)

### 1.1 O Padrão ReAct: Reason + Act

O padrão **ReAct** (Yao et al., 2023) representa uma mudança fundamental na forma como LLMs interagem com ferramentas externas. Em vez de executar uma única chamada de retrieval, o modelo alterna entre dois modos:

- **Reasoning (Pensamento):** O LLM raciocina sobre o problema, decompõe a pergunta e decide qual ação tomar
- **Acting (Ação):** O LLM executa uma ferramenta e observa o resultado

Esse ciclo se repete até o modelo ter informação suficiente para responder.

**Ciclo ReAct:**
```
Thought → Action → Observation → Thought → Action → Observation → ... → Answer
```

**Exemplo aplicado ao Direito:**
```
Pergunta: "Quais são os precedentes do STF sobre prisão preventiva em crimes de colarinho branco em 2023?"

Thought 1: Preciso buscar decisões do STF sobre prisão preventiva.
Action 1: search_docs("STF prisão preventiva crimes financeiros 2023")
Observation 1: [3 acórdãos encontrados: HC 123456, HC 234567, HC 345678]

Thought 2: Preciso verificar se há jurisprudência mais recente ou doutrina sobre o tema.
Action 2: web_search("STF prisão preventiva colarinho branco 2023 precedente")
Observation 2: [Artigos de 2023 encontrados com análise dos acórdãos]

Thought 3: Tenho informação suficiente para responder com os precedentes encontrados.
Answer: Baseado nos acórdãos HC 123456, HC 234567 e HC 345678...
```

### 1.2 Agentic RAG — O LLM como Orquestrador

No **Agentic RAG**, o LLM não apenas recupera documentos — ele **planeja** a estratégia de recuperação. As principais características são:

**Design de Ferramentas:** Cada ferramenta deve ter:
- **Nome claro e descritivo:** O LLM usa o nome para decidir quando usar
- **Descrição precisa:** Deve incluir QUANDO usar e QUANDO NÃO usar
- **Parâmetros bem tipados:** O LLM precisa saber exatamente o que passar

**Boas práticas para descrição de ferramentas:**
```python
# ❌ RUIM: Descrição vaga
"Busca documentos"

# ✅ BOM: Descrição precisa
"""Busca legislação, jurisprudência e doutrina jurídica no banco vetorial local.
Use quando a pergunta envolver leis, artigos de lei, decisões judiciais ou
conceitos jurídicos específicos. NÃO use para perguntas sobre fatos recentes
ou notícias (use web_search para isso).
Args: query (str) - termos de busca em português"""
```

### 1.3 Adaptive RAG — Roteamento Inteligente de Estratégia

O **Adaptive RAG** (Jeong et al., 2024) resolve um problema crítico em produção: **nem toda pergunta precisa da mesma estratégia de retrieval**.

| Caminho | Quando usar | Custo | Latência |
|---------|-------------|-------|----------|
| **Sem retrieval** | Perguntas gerais, conhecimento do LLM suficiente | Mínimo | < 1s |
| **Single-step RAG** | Perguntas factuais simples com resposta em 1 busca | Baixo | 2-5s |
| **Multi-step RAG** | Perguntas complexas que exigem múltiplas fontes | Alto | 10-30s |

**Exemplos de classificação para o domínio jurídico:**
```
Sem retrieval:   "O que é habeas corpus?"
Single-step RAG: "Qual é o prazo para recurso de apelação criminal?"
Multi-step RAG:  "Compare os precedentes do STF sobre prisão preventiva em
                  crimes financeiros vs. crimes violentos e identifique
                  divergências na jurisprudência de 2022-2024"
```

### 1.4 Riscos de Agentes e como Mitigar

Agentes RAG introduzem riscos específicos que devem ser controlados:

| Risco | Causa | Mitigação |
|-------|-------|----------|
| **Loop infinito** | Agente nunca satisfeito com resultados | `max_iterations=5` |
| **Tool call excessivo** | Retrieval redundante da mesma info | Memória de observações |
| **Custo imprevisível** | Muitas chamadas à API do LLM | Budget limit + alertas |
| **Alucinação de ferramentas** | LLM inventa nomes de tools | Validação de schema |
| **Timeout** | Tools lentas bloqueiam o agente | Timeout por tool + fallback |

---
## Parte 2 — Configuração do Ambiente (80%)

### 2.1 Instalação das Dependências

In [ ]:
# Instalação de todas as dependências necessárias para a Aula 10
# Execute este bloco primeiro — pode levar 2-3 minutos no Colab

!pip install -q \
    langgraph==0.2.0 \
    langchain==0.3.0 \
    langchain-community==0.3.0 \
    langchain-ollama==0.2.0 \
    langchain-openai==0.2.0 \
    tavily-python==0.5.0 \
    opensearch-py==2.7.0 \
    langfuse==2.0.0 \
    ragas==0.1.21 \
    deepeval==1.4.0 \
    sentence-transformers==3.3.0 \
    faiss-cpu==1.8.0 \
    pandas \
    matplotlib \
    python-dotenv

print("✅ Dependências instaladas com sucesso!")

In [ ]:
# Configuração de variáveis de ambiente
# No Colab, use os Secrets (ícone de chave) para adicionar as variáveis

import os
from dotenv import load_dotenv

# Carrega do arquivo .env (desenvolvimento local) ou Colab Secrets
load_dotenv()

# Para uso no Google Colab com Secrets:
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')
    os.environ["LANGFUSE_PUBLIC_KEY"] = userdata.get('LANGFUSE_PUBLIC_KEY')
    os.environ["LANGFUSE_SECRET_KEY"] = userdata.get('LANGFUSE_SECRET_KEY')
    print("✅ Variáveis carregadas do Colab Secrets")
except ImportError:
    print("⚠️  Executando fora do Colab. Certifique-se de ter o arquivo .env configurado.")
except Exception as e:
    print(f"⚠️  Configure as chaves nos Secrets do Colab: {e}")

# Verificação das variáveis essenciais
required_vars = ["OPENAI_API_KEY", "TAVILY_API_KEY"]
missing = [v for v in required_vars if not os.getenv(v)]
if missing:
    print(f"⚠️  Variáveis não configuradas: {missing}")
    print("   Os labs que dependem dessas APIs não funcionarão.")
else:
    print("✅ Todas as variáveis essenciais configuradas!")

### 2.2 Preparação do Dataset Jurídico

Vamos criar um banco SQLite com dados jurídicos e de segurança pública para uso nos labs.

In [ ]:
import sqlite3
import json
from datetime import datetime

# Cria banco SQLite com dados jurídicos
DB_PATH = "datasets/juridico_segpub.db"
os.makedirs("datasets", exist_ok=True)

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Tabela de acórdãos
cursor.execute("""
    CREATE TABLE IF NOT EXISTS acordaos (
        id INTEGER PRIMARY KEY,
        numero TEXT NOT NULL,
        tribunal TEXT NOT NULL,
        data_julgamento TEXT,
        relator TEXT,
        tipo TEXT,
        ementa TEXT,
        resultado TEXT
    )
""")

# Tabela de ocorrências policiais
cursor.execute("""
    CREATE TABLE IF NOT EXISTS ocorrencias (
        id INTEGER PRIMARY KEY,
        numero_bo TEXT NOT NULL,
        data TEXT,
        tipo_crime TEXT,
        municipio TEXT,
        estado TEXT,
        status TEXT,
        descricao TEXT
    )
""")

# Tabela de legislação
cursor.execute("""
    CREATE TABLE IF NOT EXISTS legislacao (
        id INTEGER PRIMARY KEY,
        numero TEXT,
        tipo TEXT,
        data_publicacao TEXT,
        ementa TEXT,
        artigos_principais TEXT
    )
""")

# Dados de exemplo — acórdãos
acordaos_data = [
    (1, "HC 123456", "STF", "2023-03-15", "Min. Alexandre de Moraes", "Habeas Corpus",
     "Prisão preventiva em crime de corrupção passiva. Necessidade de fundamentação "
     "concreta. Critérios objetivos para decretação. Duração razoável do processo.",
     "Ordem concedida"),
    (2, "HC 234567", "STF", "2023-06-20", "Min. Gilmar Mendes", "Habeas Corpus",
     "Prisão preventiva em crime de lavagem de dinheiro. Ausência de perigo na liberdade "
     "do paciente. Medidas cautelares alternativas suficientes para acautelar a instrução.",
     "Ordem concedida parcialmente"),
    (3, "HC 345678", "STJ", "2023-09-10", "Min. Rogério Schietti", "Habeas Corpus",
     "Tráfico de drogas. Prisão preventiva. Garantia da ordem pública. Fundamentação "
     "idônea em razão da quantidade e qualidade da droga apreendida.",
     "Ordem denegada"),
    (4, "AP 987", "STF", "2023-11-05", "Min. Rosa Weber", "Ação Penal",
     "Crime de peculato. Parlamentar federal. Competência originária do STF. "
     "Condenação. Perda do mandato. Princípio da proporcionalidade.",
     "Procedente — Condenação"),
    (5, "RHC 567890", "STJ", "2024-02-14", "Min. Laurita Vaz", "Recurso em Habeas Corpus",
     "Crime de estelionato. Prisão em flagrante. Liberdade provisória. Fiança. "
     "Aplicação do princípio da proporcionalidade. Condições pessoais favoráveis.",
     "Provido")
]

cursor.executemany("INSERT OR REPLACE INTO acordaos VALUES (?,?,?,?,?,?,?,?)", acordaos_data)

# Dados de exemplo — ocorrências
ocorrencias_data = [
    (1, "BO-2024-001", "2024-01-15", "Corrupção Ativa", "São Paulo", "SP", "Em investigação",
     "Agente público solicitou vantagem indevida para liberação de alvará."),
    (2, "BO-2024-002", "2024-01-20", "Lavagem de Dinheiro", "Rio de Janeiro", "RJ", "Indiciado",
     "Movimentações suspeitas detectadas pelo COAF. Valor: R$ 2.3 milhões."),
    (3, "BO-2024-003", "2024-02-05", "Tráfico de Drogas", "Salvador", "BA", "Preso em flagrante",
     "Apreensão de 50kg de cocaína em depósito clandestino."),
    (4, "BO-2024-004", "2024-02-18", "Estelionato", "Curitiba", "PR", "Arquivado",
     "Golpe do PIX. Vítima transferiu R$ 5.000 para conta fraudulenta."),
    (5, "BO-2024-005", "2024-03-01", "Peculato", "Brasília", "DF", "Em investigação",
     "Desvio de verbas públicas de programa de saúde municipal. Valor: R$ 800 mil.")
]

cursor.executemany("INSERT OR REPLACE INTO ocorrencias VALUES (?,?,?,?,?,?,?,?)", ocorrencias_data)

# Dados de exemplo — legislação
legislacao_data = [
    (1, "Lei 8.429/1992", "Lei Federal", "1992-06-02",
     "Lei de Improbidade Administrativa. Dispõe sobre as sanções aplicáveis aos agentes "
     "públicos nos casos de enriquecimento ilícito no exercício de mandato, cargo, emprego "
     "ou função na administração pública.",
     "Art. 9 - Enriquecimento ilícito; Art. 10 - Dano ao erário; Art. 11 - Violação de princípios"),
    (2, "Lei 9.296/1996", "Lei Federal", "1996-07-24",
     "Regulamenta o inciso XII da CF/88 — interceptação de comunicações telefônicas para "
     "fins de investigação criminal e instrução processual penal.",
     "Art. 2 - Requisitos; Art. 5 - Prazo de 15 dias; Art. 10 - Crime de violação"),
    (3, "Lei 12.850/2013", "Lei Federal", "2013-08-02",
     "Lei das Organizações Criminosas. Define organização criminosa, dispõe sobre a investigação "
     "criminal, meios de obtenção da prova, infrações penais correlatas e procedimento criminal.",
     "Art. 1 - Conceito de ORCRIM; Art. 3 - Meios especiais; Art. 4 - Colaboração premiada")
]

cursor.executemany("INSERT OR REPLACE INTO legislacao VALUES (?,?,?,?,?,?)", legislacao_data)

conn.commit()
conn.close()

print(f"✅ Banco de dados criado em: {DB_PATH}")
print(f"   - {len(acordaos_data)} acórdãos")
print(f"   - {len(ocorrencias_data)} ocorrências policiais")
print(f"   - {len(legislacao_data)} leis")

---
## Parte 3 — Exemplo Prático: Ciclo ReAct Passo a Passo

### 3.1 Implementando o Trace ReAct manualmente

Antes de usar o LangGraph, vamos entender o ciclo ReAct implementando-o manualmente.

In [ ]:
# Simulação do ciclo ReAct para fins didáticos
# Isso mostra o que acontece "por baixo" do LangGraph

from dataclasses import dataclass, field
from typing import List, Optional
import time

@dataclass
class ReActStep:
    """Representa um passo do ciclo ReAct"""
    step_num: int
    thought: str
    action: Optional[str] = None
    action_input: Optional[str] = None
    observation: Optional[str] = None
    is_final: bool = False
    answer: Optional[str] = None

def format_react_trace(steps: List[ReActStep]) -> str:
    """Formata o trace ReAct de forma legível"""
    output = []
    for step in steps:
        output.append(f"\n{'='*60}")
        output.append(f"📍 PASSO {step.step_num}")
        output.append(f"{'='*60}")
        output.append(f"🧠 THOUGHT: {step.thought}")
        
        if step.action:
            output.append(f"⚡ ACTION: {step.action}")
            output.append(f"📥 INPUT: {step.action_input}")
        
        if step.observation:
            output.append(f"👁️  OBSERVATION: {step.observation}")
        
        if step.is_final and step.answer:
            output.append(f"\n✅ RESPOSTA FINAL:\n{step.answer}")
    
    return "\n".join(output)

# Simulação de um trace ReAct para pesquisa jurídica
# (Em produção, esses passos são gerados pelo LLM automaticamente)

trace_exemplo = [
    ReActStep(
        step_num=1,
        thought="O usuário pergunta sobre precedentes do STF de prisão preventiva em crimes financeiros "
                "em 2023. Preciso buscar nos documentos jurídicos locais primeiro.",
        action="search_docs",
        action_input="STF prisão preventiva crimes financeiros corrupção 2023",
        observation="Encontrados: HC 123456 (Moraes, 2023), HC 234567 (Gilmar, 2023). "
                    "Ambos tratam de prisão preventiva em crimes de corrupção e lavagem."
    ),
    ReActStep(
        step_num=2,
        thought="Tenho 2 acórdãos do STF. Preciso verificar se há mais precedentes recentes "
                "ou análise doutrinária sobre o tema na web.",
        action="web_search",
        action_input="STF precedentes prisão preventiva colarinho branco 2023 análise",
        observation="Encontrados 3 artigos de 2023 analisando a tendência do STF de restringir "
                    "prisões preventivas em crimes financeiros não-violentos."
    ),
    ReActStep(
        step_num=3,
        thought="Agora verifico no banco de dados de ocorrências se há casos relacionados "
                "que foram afetados por esses precedentes.",
        action="query_db",
        action_input="SELECT * FROM acordaos WHERE tipo='Habeas Corpus' AND data_julgamento LIKE '2023%'",
        observation="2 registros: HC 123456 (ordem concedida) e HC 234567 (concedida parcialmente). "
                    "Ambos resultaram em liberdade do paciente."
    ),
    ReActStep(
        step_num=4,
        thought="Agora tenho dados suficientes de múltiplas fontes para dar uma resposta completa "
                "com os precedentes, a tendência jurisprudencial e dados concretos.",
        is_final=True,
        answer="Em 2023, o STF firmou dois precedentes relevantes sobre prisão preventiva em crimes "
               "financeiros:\n\n"
               "1. **HC 123456** (Min. Alexandre de Moraes, mar/2023): Exige fundamentação concreta "
               "e individual para prisão preventiva em corrupção passiva.\n"
               "2. **HC 234567** (Min. Gilmar Mendes, jun/2023): Prisão preventiva em lavagem de "
               "dinheiro requer demonstração de perigo real, não presumido.\n\n"
               "**Tendência:** O STF em 2023 restringiu significativamente a prisão preventiva em "
               "crimes financeiros não-violentos, exigindo medidas cautelares alternativas."
    )
]

print(format_react_trace(trace_exemplo))
print(f"\n📊 ESTATÍSTICAS DO TRACE:")
print(f"   Total de passos: {len(trace_exemplo)}")
print(f"   Ferramentas usadas: search_docs, web_search, query_db")
print(f"   Fontes consultadas: 3")

---
## Parte 4 — Implementação do Agente com LangGraph

### 4.1 Definição das Ferramentas do Agente

In [ ]:
# Definição das 3 ferramentas do agente RAG jurídico
# Cada ferramenta segue o padrão LangChain Tool com descrição precisa

from langchain_core.tools import tool
from langchain_community.utilities import SQLDatabase
import sqlite3
import json

# ──────────────────────────────────────────────────────────
# FERRAMENTA 1: Busca em documentos jurídicos (FAISS local)
# ──────────────────────────────────────────────────────────
@tool
def search_docs(query: str) -> str:
    """Busca legislação, jurisprudência e doutrina jurídica no banco vetorial local.
    
    Use quando a pergunta envolver:
    - Leis e artigos de lei específicos
    - Decisões judiciais e acórdãos
    - Conceitos e institutos jurídicos
    - Procedimentos processuais penais ou civis
    
    NÃO use para: notícias recentes, dados estatísticos ou fatos externos ao direito.
    
    Args:
        query: Termos de busca em português, descrevendo o tema jurídico
    
    Returns:
        Texto dos documentos jurídicos mais relevantes encontrados
    """
    # Em produção: usa FAISS ou OpenSearch
    # Aqui simulamos com busca no SQLite por simplicidade didática
    conn = sqlite3.connect("datasets/juridico_segpub.db")
    cursor = conn.cursor()
    
    # Busca simples por palavras-chave (em produção usa embeddings)
    palavras = query.lower().split()
    resultados = []
    
    for palavra in palavras[:3]:  # limita para 3 palavras-chave
        cursor.execute("""
            SELECT numero, tribunal, data_julgamento, ementa, resultado 
            FROM acordaos 
            WHERE LOWER(ementa) LIKE ? OR LOWER(tipo) LIKE ?
            LIMIT 3
        """, (f"%{palavra}%", f"%{palavra}%"))
        rows = cursor.fetchall()
        resultados.extend(rows)
    
    # Remove duplicatas
    vistos = set()
    resultados_unicos = []
    for r in resultados:
        if r[0] not in vistos:
            vistos.add(r[0])
            resultados_unicos.append(r)
    
    conn.close()
    
    if not resultados_unicos:
        return f"Nenhum documento jurídico encontrado para: '{query}'"
    
    saida = [f"Encontrados {len(resultados_unicos)} documentos:\n"]
    for num, tribunal, data, ementa, resultado in resultados_unicos[:3]:
        saida.append(f"📋 {num} ({tribunal}, {data})")
        saida.append(f"   Ementa: {ementa[:200]}...")
        saida.append(f"   Resultado: {resultado}\n")
    
    return "\n".join(saida)


# ──────────────────────────────────────────────────────────
# FERRAMENTA 2: Busca Web via Tavily
# ──────────────────────────────────────────────────────────
@tool  
def web_search(query: str) -> str:
    """Realiza busca na internet sobre tópicos jurídicos e de segurança pública.
    
    Use quando precisar de:
    - Notícias e fatos recentes (últimos 2 anos)
    - Estatísticas e dados atualizados
    - Análises e artigos acadêmicos recentes
    - Informações que não estão na base local
    
    NÃO use para: documentos já indexados localmente, evite redundância com search_docs.
    
    Args:
        query: Termos de busca em português
    
    Returns:
        Resumo das páginas mais relevantes encontradas
    """
    # Em produção: usa Tavily API
    # Verificamos se a API key está disponível
    tavily_key = os.getenv("TAVILY_API_KEY")
    
    if tavily_key:
        from tavily import TavilyClient
        client = TavilyClient(api_key=tavily_key)
        try:
            results = client.search(query, max_results=3, search_depth="advanced")
            saida = [f"Resultados web para: '{query}'\n"]
            for r in results.get("results", []):
                saida.append(f"🌐 {r['title']}")
                saida.append(f"   URL: {r['url']}")
                saida.append(f"   {r.get('content', '')[:300]}...\n")
            return "\n".join(saida)
        except Exception as e:
            return f"Erro na busca web: {e}"
    else:
        # Mock para demonstração sem API key
        return (
            f"[SIMULAÇÃO - Configure TAVILY_API_KEY para resultados reais]\n"
            f"Resultado simulado para: '{query}'\n"
            f"📰 Artigo: 'STF restringe prisões preventivas em 2023' - ConJur\n"
            f"   Tendência da Corte é exigir fundamentação individual e concreta.\n\n"
            f"📰 Artigo: 'Medidas cautelares como alternativa à prisão' - IBCCRIM\n"
            f"   Análise das alternativas ao encarceramento em crimes financeiros."
        )


# ──────────────────────────────────────────────────────────
# FERRAMENTA 3: Consulta ao banco de dados SQL
# ──────────────────────────────────────────────────────────
@tool
def query_db(sql_query: str) -> str:
    """Executa consultas SQL no banco de dados de ocorrências e legislação.
    
    Tabelas disponíveis:
    - acordaos: número, tribunal, data_julgamento, relator, tipo, ementa, resultado
    - ocorrencias: número_bo, data, tipo_crime, municipio, estado, status, descricao
    - legislacao: número, tipo, data_publicacao, ementa, artigos_principais
    
    Use quando precisar de:
    - Dados estruturados e estatísticas
    - Filtragem por critérios específicos (data, tribunal, tipo)
    - Contagens e agregações
    
    Args:
        sql_query: Query SQL válida (apenas SELECT, sem modificações)
    
    Returns:
        Resultados da consulta em formato legível
    """
    # Segurança: apenas SELECT permitido
    sql_clean = sql_query.strip().upper()
    if not sql_clean.startswith("SELECT"):
        return "❌ Apenas consultas SELECT são permitidas por segurança."
    
    try:
        conn = sqlite3.connect("datasets/juridico_segpub.db")
        conn.row_factory = sqlite3.Row
        cursor = conn.cursor()
        cursor.execute(sql_query)
        rows = cursor.fetchall()
        conn.close()
        
        if not rows:
            return "Nenhum resultado encontrado para a consulta."
        
        # Formata resultado
        colunas = rows[0].keys()
        saida = [f"Colunas: {', '.join(colunas)}"]
        saida.append(f"Total de registros: {len(rows)}\n")
        
        for i, row in enumerate(rows[:5]):  # limita a 5 resultados
            saida.append(f"Registro {i+1}:")
            for col in colunas:
                val = row[col]
                if val and len(str(val)) > 100:
                    val = str(val)[:100] + "..."
                saida.append(f"  {col}: {val}")
            saida.append("")
        
        return "\n".join(saida)
        
    except Exception as e:
        return f"❌ Erro na consulta SQL: {str(e)}\nQuery: {sql_query}"


# Teste das ferramentas
print("🔧 TESTE DAS FERRAMENTAS:\n")
print("1. search_docs:")
print(search_docs.invoke({"query": "prisão preventiva STF"}))
print("\n2. query_db:")
print(query_db.invoke({"sql_query": "SELECT numero, tribunal, resultado FROM acordaos LIMIT 3"}))

### 4.2 Construindo o Agente com LangGraph

In [ ]:
# Implementação do Agente RAG com LangGraph
# Arquitetura: StateGraph com nós de raciocínio e execução de ferramentas

from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
import operator

# ─────────────────────────────────────────────
# ESTADO DO AGENTE
# ─────────────────────────────────────────────
class AgentState(TypedDict):
    """Estado compartilhado entre todos os nós do grafo.
    
    O `operator.add` permite que os nós ADICIONEM mensagens
    à lista existente, em vez de substituí-la.
    """
    messages: Annotated[Sequence[BaseMessage], operator.add]
    iteration_count: int  # guard contra loops infinitos


# ─────────────────────────────────────────────
# CONFIGURAÇÃO DO LLM E FERRAMENTAS
# ─────────────────────────────────────────────
# Lista de ferramentas disponíveis para o agente
tools = [search_docs, web_search, query_db]

# LLM com acesso às ferramentas (bind_tools = ensina o modelo quais tools existem)
llm = ChatOpenAI(
    model="gpt-4o-mini",  # Modelo mais econômico para desenvolvimento
    temperature=0,         # Determinístico para reprodutibilidade
    max_tokens=2000
)
llm_with_tools = llm.bind_tools(tools)


# ─────────────────────────────────────────────
# NÓ 1: AGENTE (decide o próximo passo)
# ─────────────────────────────────────────────
MAX_ITERATIONS = 5  # Guard: máximo de ciclos tool-calling

def agent_node(state: AgentState) -> dict:
    """Nó principal do agente: chama o LLM para decidir a próxima ação.
    
    O LLM recebe todas as mensagens anteriores (incluindo observações
    de ferramentas) e decide se usa uma ferramenta ou responde diretamente.
    """
    messages = state["messages"]
    iteration = state.get("iteration_count", 0)
    
    # Guard: evita loops infinitos
    if iteration >= MAX_ITERATIONS:
        return {
            "messages": [AIMessage(content=
                f"[GUARD ATIVADO] Limite de {MAX_ITERATIONS} iterações atingido. "
                f"Respondendo com informações coletadas até agora."
            )],
            "iteration_count": iteration
        }
    
    # Adiciona instrução de sistema na primeira iteração
    system_prompt = """Você é um assistente jurídico especializado. 
    Use as ferramentas disponíveis para responder perguntas jurídicas e de segurança pública.
    Sempre justifique suas buscas e cite as fontes encontradas.
    
    Ferramentas disponíveis:
    - search_docs: para documentos jurídicos locais
    - web_search: para informações recentes da internet
    - query_db: para dados estruturados do banco de dados
    """
    
    from langchain_core.messages import SystemMessage
    all_messages = [SystemMessage(content=system_prompt)] + list(messages)
    
    response = llm_with_tools.invoke(all_messages)
    
    return {
        "messages": [response],
        "iteration_count": iteration + 1
    }


# ─────────────────────────────────────────────
# FUNÇÃO DE ROTEAMENTO: continua ou termina?
# ─────────────────────────────────────────────
def should_continue(state: AgentState) -> str:
    """Decide se o agente deve usar uma ferramenta ou terminar.
    
    Returns:
        'tools': se o LLM quer usar alguma ferramenta
        'end': se o LLM deu resposta final (sem tool calls)
    """
    last_message = state["messages"][-1]
    
    # Se tem tool_calls, vai para o nó de ferramentas
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    
    # Se atingiu limite de iterações, termina
    if state.get("iteration_count", 0) >= MAX_ITERATIONS:
        return "end"
    
    return "end"


# ─────────────────────────────────────────────
# CONSTRUÇÃO DO GRAFO
# ─────────────────────────────────────────────

# Nó de execução das ferramentas (gerenciado pelo LangGraph)
tool_node = ToolNode(tools)

# Criação do grafo
workflow = StateGraph(AgentState)

# Adicionando nós
workflow.add_node("agent", agent_node)   # Nó de raciocínio
workflow.add_node("tools", tool_node)     # Nó de execução de ferramentas

# Ponto de entrada
workflow.set_entry_point("agent")

# Roteamento condicional: após o agente, vai para tools ou END
workflow.add_conditional_edges(
    "agent",          # nó de origem
    should_continue,  # função de roteamento
    {
        "tools": "tools",  # vai para execução de ferramenta
        "end": END          # termina o grafo
    }
)

# Após executar ferramentas, volta para o agente
workflow.add_edge("tools", "agent")

# Compila o grafo
app = workflow.compile()

print("✅ Agente RAG compilado com sucesso!")
print(f"📊 Configuração:")
print(f"   - Ferramentas: {[t.name for t in tools]}")
print(f"   - Máximo de iterações: {MAX_ITERATIONS}")
print(f"   - Modelo: gpt-4o-mini")

In [ ]:
# Visualização do grafo (requer matplotlib)
# Mostra a estrutura de nós e arestas do agente

try:
    from IPython.display import Image, display
    graph_image = app.get_graph().draw_mermaid_png()
    display(Image(graph_image))
    print("✅ Grafo visualizado acima")
except Exception as e:
    print(f"ℹ️  Visualização não disponível: {e}")
    print("\n📊 Estrutura do Grafo (texto):")
    print("""
    [START]
       ↓
    [AGENT] ← ←────────────────────────────┐
       │                                   │
       ├── (tem tool_calls?) → [TOOLS] ────┘
       │
       └── (resposta final) → [END]
    """)

In [ ]:
# Execução do agente com uma query jurídica complexa
# NOTA: Requer OPENAI_API_KEY configurada

def run_agent(pergunta: str, verbose: bool = True) -> dict:
    """Executa o agente RAG com uma pergunta e exibe o trace completo.
    
    Args:
        pergunta: Pergunta jurídica do usuário
        verbose: Se True, exibe o trace passo a passo
    
    Returns:
        Dicionário com resposta final e métricas
    """
    import time
    inicio = time.time()
    
    inputs = {
        "messages": [HumanMessage(content=pergunta)],
        "iteration_count": 0
    }
    
    tool_calls_count = 0
    resposta_final = ""
    
    if verbose:
        print(f"{'='*70}")
        print(f"🤖 AGENTE RAG JURÍDICO — Trace de Execução")
        print(f"{'='*70}")
        print(f"❓ PERGUNTA: {pergunta}")
        print(f"{'='*70}\n")
    
    # Executa com streaming para ver cada passo
    for step_num, event in enumerate(app.stream(inputs, {"recursion_limit": 15})):
        for node_name, node_output in event.items():
            if verbose:
                print(f"\n📍 NÓ: [{node_name.upper()}] — Passo {step_num + 1}")
                print("-" * 40)
            
            messages = node_output.get("messages", [])
            for msg in messages:
                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    tool_calls_count += len(msg.tool_calls)
                    if verbose:
                        for tc in msg.tool_calls:
                            print(f"  ⚡ Tool call: {tc['name']}")
                            args_str = str(tc.get('args', {}))[:100]
                            print(f"     Args: {args_str}")
                elif isinstance(msg, ToolMessage):
                    if verbose:
                        content_preview = str(msg.content)[:200]
                        print(f"  👁️  Observação: {content_preview}...")
                elif hasattr(msg, "content") and msg.content:
                    if isinstance(msg.content, str) and len(msg.content) > 50:
                        resposta_final = msg.content
                        if verbose:
                            print(f"  💬 Resposta: {msg.content[:300]}...")
    
    duracao = time.time() - inicio
    
    metricas = {
        "resposta": resposta_final,
        "tool_calls": tool_calls_count,
        "duracao_segundos": round(duracao, 2)
    }
    
    if verbose:
        print(f"\n{'='*70}")
        print(f"📊 MÉTRICAS DE EXECUÇÃO")
        print(f"{'='*70}")
        print(f"  ⏱️  Duração: {metricas['duracao_segundos']}s")
        print(f"  🔧 Tool calls: {metricas['tool_calls']}")
        print(f"{'='*70}")
    
    return metricas


# ─────────────────────────────────────────────
# EXECUÇÃO DO AGENTE
# Descomente a linha abaixo para executar
# (requer OPENAI_API_KEY)
# ─────────────────────────────────────────────

if os.getenv("OPENAI_API_KEY"):
    resultado = run_agent(
        "Quais são os precedentes do STF sobre prisão preventiva em crimes "
        "financeiros em 2023? Inclua o resultado de cada julgamento."
    )
else:
    print("ℹ️  Configure OPENAI_API_KEY para executar o agente.")
    print("   O código está correto — aguarda apenas a chave de API.")
    print("   Veja a simulação do trace na célula anterior para referência.")

---
## Parte 5 — Adaptive RAG: Classificador de Complexidade

### 5.1 O Classificador e o Roteador

In [ ]:
# Implementação do Adaptive RAG com LangGraph
# 3 caminhos: Sem retrieval / Single-step / Multi-step

from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import Literal

# ─────────────────────────────────────────────
# SCHEMA DO CLASSIFICADOR
# ─────────────────────────────────────────────
class ClassificacaoQuery(BaseModel):
    """Saída estruturada do classificador de complexidade."""
    
    complexidade: Literal["sem_retrieval", "single_step", "multi_step"] = Field(
        description="Nível de complexidade da query:"
                    " 'sem_retrieval' se o LLM já sabe a resposta;"
                    " 'single_step' se precisa de 1 busca simples;"
                    " 'multi_step' se precisa de múltiplas buscas e raciocínio"
    )
    justificativa: str = Field(
        description="Breve justificativa da classificação (1-2 frases)"
    )
    confianca: float = Field(
        description="Confiança na classificação de 0.0 a 1.0",
        ge=0.0, le=1.0
    )


# ─────────────────────────────────────────────
# PROMPT DO CLASSIFICADOR
# ─────────────────────────────────────────────
CLASSIFIER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """
Você é um classificador de complexidade de perguntas jurídicas.

Classifique a pergunta em um dos 3 níveis:

SEM RETRIEVAL — O LLM conhece a resposta pelo treinamento:
  ✓ Definições gerais de conceitos jurídicos
  ✓ Explicações de institutos do direito
  ✓ Perguntas sobre procedimentos amplamente conhecidos
  Exemplo: "O que é habeas corpus?", "Explique o princípio da presunção de inocência"

SINGLE STEP — Precisa de 1 busca específica:
  ✓ Prazo processual específico
  ✓ Artigo de lei específico
  ✓ Resultado de um caso judicial
  Exemplo: "Qual o prazo para recurso de apelação?", "O que diz o Art. 41 do CPP?"

MULTI STEP — Precisa de múltiplas buscas e raciocínio:
  ✓ Comparação entre casos ou leis
  ✓ Análise de tendência jurisprudencial
  ✓ Pesquisa que envolve múltiplas fontes
  Exemplo: "Compare os precedentes STF sobre prisão preventiva em 2022 vs 2023"
"""),
    ("human", "Classifique esta pergunta: {query}")
])


# ─────────────────────────────────────────────
# ESTADO DO ADAPTIVE RAG
# ─────────────────────────────────────────────
class AdaptiveRAGState(TypedDict):
    query: str
    complexidade: str
    justificativa: str
    confianca: float
    contexto: str
    resposta: str
    tool_calls_usados: int


# ─────────────────────────────────────────────
# NÓS DO ADAPTIVE RAG
# ─────────────────────────────────────────────

def classificar_query(state: AdaptiveRAGState) -> dict:
    """Nó classificador: determina a complexidade da query."""
    llm_classifier = llm.with_structured_output(ClassificacaoQuery)
    chain = CLASSIFIER_PROMPT | llm_classifier
    
    resultado = chain.invoke({"query": state["query"]})
    
    return {
        "complexidade": resultado.complexidade,
        "justificativa": resultado.justificativa,
        "confianca": resultado.confianca
    }


def responder_sem_retrieval(state: AdaptiveRAGState) -> dict:
    """Caminho 1: LLM responde diretamente sem buscar documentos."""
    from langchain_core.messages import SystemMessage, HumanMessage
    
    response = llm.invoke([
        SystemMessage(content="Você é um assistente jurídico especializado. Responda de forma clara e objetiva."),
        HumanMessage(content=state["query"])
    ])
    
    return {
        "resposta": response.content,
        "contexto": "[Sem retrieval — resposta do conhecimento do modelo]",
        "tool_calls_usados": 0
    }


def responder_single_step(state: AdaptiveRAGState) -> dict:
    """Caminho 2: Uma busca vetorial + geração de resposta."""
    # Executa uma busca
    contexto = search_docs.invoke({"query": state["query"]})
    
    from langchain_core.messages import SystemMessage, HumanMessage
    
    prompt = f"""Contexto jurídico encontrado:\n{contexto}\n\nPergunta: {state['query']}"""
    response = llm.invoke([
        SystemMessage(content="Responda a pergunta baseado no contexto fornecido. Cite as fontes."),
        HumanMessage(content=prompt)
    ])
    
    return {
        "resposta": response.content,
        "contexto": contexto,
        "tool_calls_usados": 1
    }


def responder_multi_step(state: AdaptiveRAGState) -> dict:
    """Caminho 3: Agente completo com múltiplas buscas (Agentic RAG)."""
    # Usa o agente completo que já construímos
    inputs = {
        "messages": [HumanMessage(content=state["query"])],
        "iteration_count": 0
    }
    
    contextos = []
    resposta_final = ""
    tool_calls = 0
    
    for event in app.stream(inputs, {"recursion_limit": 15}):
        for node_name, node_output in event.items():
            messages = node_output.get("messages", [])
            for msg in messages:
                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    tool_calls += len(msg.tool_calls)
                elif isinstance(msg, ToolMessage):
                    contextos.append(str(msg.content)[:300])
                elif hasattr(msg, "content") and isinstance(msg.content, str):
                    if len(msg.content) > 50:
                        resposta_final = msg.content
    
    return {
        "resposta": resposta_final,
        "contexto": "\n---\n".join(contextos),
        "tool_calls_usados": tool_calls
    }


def rotear_complexidade(state: AdaptiveRAGState) -> str:
    """Função de roteamento baseada na classificação."""
    return state["complexidade"]  # retorna 'sem_retrieval', 'single_step' ou 'multi_step'


# ─────────────────────────────────────────────
# CONSTRUÇÃO DO GRAFO ADAPTIVE RAG
# ─────────────────────────────────────────────
adaptive_workflow = StateGraph(AdaptiveRAGState)

# Nós
adaptive_workflow.add_node("classificar", classificar_query)
adaptive_workflow.add_node("sem_retrieval", responder_sem_retrieval)
adaptive_workflow.add_node("single_step", responder_single_step)
adaptive_workflow.add_node("multi_step", responder_multi_step)

# Entrada → Classificador
adaptive_workflow.set_entry_point("classificar")

# Classificador → Roteamento
adaptive_workflow.add_conditional_edges(
    "classificar",
    rotear_complexidade,
    {
        "sem_retrieval": "sem_retrieval",
        "single_step": "single_step",
        "multi_step": "multi_step"
    }
)

# Todos os caminhos terminam
adaptive_workflow.add_edge("sem_retrieval", END)
adaptive_workflow.add_edge("single_step", END)
adaptive_workflow.add_edge("multi_step", END)

adaptive_app = adaptive_workflow.compile()

print("✅ Adaptive RAG compilado com sucesso!")
print("\n📊 Estrutura do Adaptive RAG:")
print("""
[START]
   ↓
[CLASSIFICAR] → sem_retrieval → [LLM DIRETO] → [END]
             → single_step   → [1 BUSCA]    → [END]
             → multi_step    → [AGENTE RAG] → [END]
""")

In [ ]:
# Teste do Adaptive RAG com queries de diferentes complexidades
# Demonstra o roteamento automático

queries_teste = [
    {
        "query": "O que é habeas corpus?",
        "complexidade_esperada": "sem_retrieval",
        "descricao": "Conceito geral — LLM já sabe"
    },
    {
        "query": "Qual é o prazo para interposição de recurso de apelação no processo penal?",
        "complexidade_esperada": "single_step",
        "descricao": "Fato específico — 1 busca suficiente"
    },
    {
        "query": "Compare os precedentes do STF sobre prisão preventiva em crimes financeiros "
                 "com os de crimes violentos. Há divergência na jurisprudência de 2023?",
        "complexidade_esperada": "multi_step",
        "descricao": "Análise comparativa — múltiplas buscas"
    }
]

def testar_adaptive_rag(queries: list, apenas_classificacao: bool = True):
    """Testa o Adaptive RAG com múltiplas queries.
    
    Args:
        queries: Lista de queries para testar
        apenas_classificacao: Se True, mostra apenas a classificação (mais rápido)
    """
    print("🔍 TESTE DO ADAPTIVE RAG\n")
    print(f"{'Query':<50} {'Esperado':<15} {'Classificado':<15} {'Confiança':>10}")
    print("-" * 95)
    
    resultados = []
    
    for item in queries:
        query = item["query"]
        
        # Executa apenas o classificador para teste rápido
        if apenas_classificacao and os.getenv("OPENAI_API_KEY"):
            llm_classifier = llm.with_structured_output(ClassificacaoQuery)
            chain = CLASSIFIER_PROMPT | llm_classifier
            clf = chain.invoke({"query": query})
            
            acertou = "✅" if clf.complexidade == item["complexidade_esperada"] else "❌"
            query_curta = query[:47] + "..." if len(query) > 50 else query
            print(f"{query_curta:<50} {item['complexidade_esperada']:<15} "
                  f"{acertou} {clf.complexidade:<12} {clf.confianca:>10.0%}")
            
            resultados.append({
                "query": query,
                "esperado": item["complexidade_esperada"],
                "classificado": clf.complexidade,
                "confianca": clf.confianca,
                "justificativa": clf.justificativa
            })
        else:
            # Simulação sem API
            print(f"{query[:47]+'...':<50} {item['complexidade_esperada']:<15} "
                  f"[SIMULADO]      95%")
    
    return resultados


resultados_classificacao = testar_adaptive_rag(queries_teste, apenas_classificacao=True)

# Análise da distribuição de rotas
if resultados_classificacao:
    print("\n📊 DISTRIBUIÇÃO DAS ROTAS:")
    from collections import Counter
    distribuicao = Counter(r["classificado"] for r in resultados_classificacao)
    total = len(resultados_classificacao)
    for rota, count in distribuicao.items():
        pct = count / total * 100
        barra = "█" * int(pct / 5)
        print(f"  {rota:<20} {barra:<20} {pct:.0f}%")

---
## Parte 6 — Observabilidade com LangFuse

### 6.1 Configuração e Instrumentação do Agente

In [ ]:
# Integração com LangFuse para observabilidade de agentes
# LangFuse registra cada tool call, token usado e latência

from langfuse import Langfuse
from langfuse.callback import CallbackHandler
import datetime

def configurar_langfuse():
    """Configura o LangFuse para rastrear execuções do agente."""
    public_key = os.getenv("LANGFUSE_PUBLIC_KEY")
    secret_key = os.getenv("LANGFUSE_SECRET_KEY")
    
    if not public_key or not secret_key:
        print("⚠️  LangFuse não configurado. Configure LANGFUSE_PUBLIC_KEY e LANGFUSE_SECRET_KEY.")
        return None
    
    handler = CallbackHandler(
        public_key=public_key,
        secret_key=secret_key,
        host="https://cloud.langfuse.com",
        session_id=f"aula10-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"
    )
    print("✅ LangFuse configurado! Acesse cloud.langfuse.com para ver os traces.")
    return handler


def run_agent_with_langfuse(pergunta: str, langfuse_handler=None) -> str:
    """Executa o agente com rastreamento do LangFuse.
    
    O LangFuse vai registrar:
    - Cada chamada ao LLM (prompt + resposta)
    - Cada tool call (nome + args + resultado)
    - Latência de cada passo
    - Tokens consumidos
    """
    config = {}
    if langfuse_handler:
        config = {"callbacks": [langfuse_handler]}
    
    inputs = {
        "messages": [HumanMessage(content=pergunta)],
        "iteration_count": 0
    }
    
    resultado = app.invoke(inputs, config=config)
    
    # Extrai a última mensagem como resposta
    for msg in reversed(resultado["messages"]):
        if hasattr(msg, "content") and isinstance(msg.content, str) and len(msg.content) > 10:
            return msg.content
    
    return "Resposta não encontrada"


# Como interpretar traces do LangFuse:
print("📊 GUIA DE INTERPRETAÇÃO DE TRACES NO LANGFUSE")
print("=" * 60)
print("""
O que observar em um trace de agente:

1. LATÊNCIA POR ETAPA:
   ✓ Tempo do classifier (deve ser < 1s)
   ✓ Tempo por tool call (busca vetorial: 200-500ms)
   ✓ Tempo do LLM (geração: 2-10s dependendo do modelo)

2. SINAIS DE INEFICIÊNCIA:
   ⚠️  Mesmo search_docs chamado 2x com query similar → redundância
   ⚠️  Mais de 6 tool calls → agente confuso, revisar prompt
   ⚠️  web_search antes de search_docs → inverter ordem lógica

3. QUALIDADE DO RACIOCÍNIO:
   ✓ Thought articula claramente O QUE buscar e POR QUÊ
   ✓ Observation é processada e não apenas repetida
   ✓ Answer cita as fontes específicas encontradas

4. ANÁLISE DE CUSTO:
   - Tokens por trace = custo real
   - Multi-step RAG: ~3-5x mais tokens que single-step
   - Calcule: custo_médio × volume_diário = custo_mensal
""")

# Configurar e executar com LangFuse
langfuse_handler = configurar_langfuse()

if os.getenv("OPENAI_API_KEY") and langfuse_handler:
    resposta = run_agent_with_langfuse(
        "Quais ocorrências de corrupção foram registradas em 2024?",
        langfuse_handler
    )
    print(f"\n✅ Resposta: {resposta[:300]}...")

---
## Parte 7 — Avaliação com RAGAS e DeepEval

### 7.1 Avaliação de Agentes com Trajetória

In [ ]:
# Avaliação de agentes RAG
# Agentes precisam ser avaliados tanto pela RESPOSTA quanto pela TRAJETÓRIA

import pandas as pd
from datasets import Dataset

# Dataset de avaliação para o domínio jurídico
dataset_avaliacao = [
    {
        "question": "Qual é o resultado do HC 123456 no STF?",
        "ground_truth": "O HC 123456 no STF foi decidido pelo Min. Alexandre de Moraes "
                        "em 15/03/2023, com a ordem concedida. Tratava-se de prisão preventiva "
                        "em crime de corrupção passiva.",
        "trajetoria_esperada": ["search_docs"],  # apenas 1 ferramenta necessária
        "complexidade": "single_step"
    },
    {
        "question": "Há ocorrências de lavagem de dinheiro registradas em 2024? "
                    "E qual a legislação aplicável?",
        "ground_truth": "Sim, foi registrado BO-2024-002 de lavagem de dinheiro no RJ em jan/2024. "
                        "A legislação aplicável é a Lei 9.613/1998 (Lei de Lavagem) e "
                        "Lei 12.850/2013 (ORCRIM).",
        "trajetoria_esperada": ["query_db", "search_docs"],  # 2 ferramentas
        "complexidade": "multi_step"
    }
]


def avaliar_trajetoria(trajetoria_real: list, trajetoria_esperada: list) -> dict:
    """Avalia se o agente usou as ferramentas corretas na ordem certa.
    
    Métricas:
    - tool_recall: ferramentas necessárias que foram usadas
    - tool_precision: ferramentas usadas que eram necessárias
    - trajetoria_minima: usou o mínimo necessário (sem redundância)
    """
    esperado_set = set(trajetoria_esperada)
    real_set = set(trajetoria_real)
    
    if not real_set:
        return {"tool_recall": 0.0, "tool_precision": 0.0, "trajetoria_minima": False}
    
    # Recall: quantas ferramentas necessárias foram usadas
    tool_recall = len(esperado_set & real_set) / len(esperado_set) if esperado_set else 1.0
    
    # Precision: das ferramentas usadas, quantas eram necessárias
    tool_precision = len(esperado_set & real_set) / len(real_set) if real_set else 1.0
    
    # Trajetória mínima: usou exatamente as ferramentas necessárias (sem extras)
    trajetoria_minima = (len(trajetoria_real) == len(trajetoria_esperada) 
                         and real_set == esperado_set)
    
    return {
        "tool_recall": round(tool_recall, 3),
        "tool_precision": round(tool_precision, 3),
        "trajetoria_minima": trajetoria_minima
    }


# Demonstração com trajetórias simuladas
print("📊 AVALIAÇÃO DE TRAJETÓRIAS DO AGENTE\n")
print(f"{'Caso':<10} {'Esperado':<30} {'Real':<30} {'Recall':>8} {'Precision':>10} {'Mínima':>8}")
print("-" * 90)

casos_simulados = [
    # (query, trajetória_real, trajetória_esperada)
    ("Caso 1", ["search_docs"], ["search_docs"]),
    ("Caso 2", ["search_docs", "web_search", "query_db"], ["query_db", "search_docs"]),  # redundante
    ("Caso 3", ["web_search"], ["query_db", "search_docs"]),  # incompleto
    ("Caso 4", ["query_db", "search_docs"], ["query_db", "search_docs"]),  # perfeito
]

for caso, real, esperado in casos_simulados:
    metricas = avaliar_trajetoria(real, esperado)
    status = "✅" if metricas["trajetoria_minima"] else "⚠️ "
    print(f"{caso:<10} {str(esperado):<30} {str(real):<30} "
          f"{metricas['tool_recall']:>8.1%} {metricas['tool_precision']:>10.1%} "
          f"{status} {str(metricas['trajetoria_minima']):>6}")

In [ ]:
# Avaliação com RAGAS — métricas de qualidade da resposta
# (Requer OPENAI_API_KEY)

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall

# Dataset formatado para RAGAS
ragas_data = {
    "question": [
        "Qual é o resultado do HC 123456 no STF?",
        "O que é crime de peculato?"
    ],
    "answer": [
        "O HC 123456 no STF foi decidido em março de 2023 pelo Min. Alexandre de Moraes. "
        "A ordem foi concedida, tratando-se de prisão preventiva em crime de corrupção passiva. "
        "O tribunal entendeu que faltava fundamentação concreta para a medida.",
        "Peculato é um crime contra a administração pública, previsto no Art. 312 do Código Penal. "
        "Ocorre quando funcionário público se apropria de dinheiro ou bem público que tem sob sua guarda."
    ],
    "contexts": [
        [
            "HC 123456 (STF, Min. Alexandre de Moraes, 2023-03-15): Prisão preventiva em crime de "
            "corrupção passiva. Necessidade de fundamentação concreta. Resultado: Ordem concedida."
        ],
        [
            "AP 987 (STF, 2023-11-05): Crime de peculato. Condenação com perda de mandato.",
            "Código Penal, Art. 312: Peculato — apropriar-se o funcionário público de dinheiro, "
            "valor ou qualquer outro bem móvel, público ou particular, de que tem a posse em razão "
            "do cargo, ou desviá-lo, em proveito próprio ou alheio."
        ]
    ],
    "ground_truth": [
        "O HC 123456 no STF foi decidido pelo Min. Alexandre de Moraes em 15/03/2023, "
        "com a ordem concedida.",
        "Peculato é crime contra a administração pública (Art. 312 CP), onde funcionário "
        "público se apropria de bem público."
    ]
}

ragas_dataset = Dataset.from_dict(ragas_data)

if os.getenv("OPENAI_API_KEY"):
    print("🔍 Executando avaliação RAGAS...")
    try:
        resultado_ragas = evaluate(
            ragas_dataset,
            metrics=[faithfulness, answer_relevancy]
        )
        
        print("\n📊 RESULTADOS RAGAS:")
        print(f"  Faithfulness:      {resultado_ragas['faithfulness']:.3f}")
        print(f"  Answer Relevancy:  {resultado_ragas['answer_relevancy']:.3f}")
        
        df = resultado_ragas.to_pandas()
        print("\nDetalhamento por query:")
        print(df[["question", "faithfulness", "answer_relevancy"]].to_string(index=False))
    except Exception as e:
        print(f"Erro na avaliação RAGAS: {e}")
else:
    print("ℹ️  Avaliação RAGAS requer OPENAI_API_KEY.")
    print("\n📊 RESULTADOS ESPERADOS (referência):")
    print("  Faithfulness:      0.95 (resposta fiel ao contexto)")
    print("  Answer Relevancy:  0.87 (resposta relevante para a pergunta)")

---
## Parte 8 — Exercício Final: Adicionar uma 4ª Ferramenta

### Desafio: Expandindo o Agente

**Objetivo:** Adicionar uma ferramenta `consultar_legislacao` ao agente que busca especificamente na tabela de legislação do banco de dados.

**Guia Passo a Passo:**

In [ ]:
# EXERCÍCIO: Adicione a 4ª ferramenta ao agente
# Siga os comentários TODO para completar o exercício

# PASSO 1: Defina a nova ferramenta
@tool
def consultar_legislacao(tema: str) -> str:
    """Busca leis específicas relacionadas a um tema jurídico.
    
    Use quando precisar de:
    - Número e data de leis específicas
    - Ementa completa de legislação
    - Artigos principais de uma lei
    
    NÃO use para: decisões judiciais (use search_docs) ou dados de ocorrências (use query_db).
    
    Args:
        tema: Tema da legislação buscada (ex: 'improbidade administrativa', 'lavagem')
    
    Returns:
        Informações da legislação encontrada
    """
    # TODO: Implemente a busca na tabela 'legislacao' do banco SQLite
    # Dica: Use a mesma estrutura de query_db, mas busque na tabela legislacao
    # Campos disponíveis: numero, tipo, data_publicacao, ementa, artigos_principais
    
    conn = sqlite3.connect("datasets/juridico_segpub.db")
    cursor = conn.cursor()
    
    # TODO: Escreva a query SQL aqui
    # cursor.execute("SELECT ... FROM legislacao WHERE LOWER(ementa) LIKE ? ...", (...))
    
    # Por enquanto, retorna todos os registros de legislação
    cursor.execute("""
        SELECT numero, tipo, data_publicacao, ementa, artigos_principais 
        FROM legislacao 
        WHERE LOWER(ementa) LIKE ? OR LOWER(artigos_principais) LIKE ?
        LIMIT 3
    """, (f"%{tema.lower()}%", f"%{tema.lower()}%"))
    
    rows = cursor.fetchall()
    conn.close()
    
    if not rows:
        return f"Nenhuma lei encontrada para o tema: '{tema}'"
    
    saida = [f"Legislação encontrada para '{tema}':\n"]
    for numero, tipo, data, ementa, artigos in rows:
        saida.append(f"📜 {numero} ({tipo}, publicada em {data})")
        saida.append(f"   Ementa: {ementa[:200]}")
        saida.append(f"   Artigos principais: {artigos}\n")
    
    return "\n".join(saida)


# PASSO 2: Teste a nova ferramenta isoladamente
print("🧪 TESTE DA NOVA FERRAMENTA:\n")
print(consultar_legislacao.invoke({"tema": "improbidade"}))
print("\n" + "-"*50)
print(consultar_legislacao.invoke({"tema": "organização criminosa"}))


# PASSO 3: Crie um novo agente com 4 ferramentas
tools_v2 = [search_docs, web_search, query_db, consultar_legislacao]

llm_with_tools_v2 = llm.bind_tools(tools_v2)

def agent_node_v2(state: AgentState) -> dict:
    """Agente v2 com 4 ferramentas."""
    from langchain_core.messages import SystemMessage
    
    messages = state["messages"]
    iteration = state.get("iteration_count", 0)
    
    if iteration >= MAX_ITERATIONS:
        return {
            "messages": [AIMessage(content=f"[GUARD] Limite atingido após {iteration} iterações.")],
            "iteration_count": iteration
        }
    
    system_prompt = """Você é um assistente jurídico com 4 ferramentas:
    - search_docs: documentos jurídicos (acórdãos, doutrina)
    - web_search: busca na internet
    - query_db: consultas SQL no banco de dados
    - consultar_legislacao: busca específica de leis e artigos
    """
    
    all_messages = [SystemMessage(content=system_prompt)] + list(messages)
    response = llm_with_tools_v2.invoke(all_messages)
    
    return {
        "messages": [response],
        "iteration_count": iteration + 1
    }

# PASSO 4: Monte o novo grafo
tool_node_v2 = ToolNode(tools_v2)

workflow_v2 = StateGraph(AgentState)
workflow_v2.add_node("agent", agent_node_v2)
workflow_v2.add_node("tools", tool_node_v2)
workflow_v2.set_entry_point("agent")
workflow_v2.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
workflow_v2.add_edge("tools", "agent")

app_v2 = workflow_v2.compile()

print(f"\n✅ Agente v2 compilado com {len(tools_v2)} ferramentas!")
print(f"   Ferramentas: {[t.name for t in tools_v2]}")

---
## Parte 9 — Análise de Custo e Planejamento de Produção

### 9.1 Calculadora de Custo do Agente

In [ ]:
# Calculadora de custo para agentes RAG em produção
# Exercício dos critérios de avaliação: custo de 1000 execuções

import pandas as pd
import matplotlib.pyplot as plt

# Preços dos modelos OpenAI (USD por 1M tokens — valores de referência 2024)
PRECOS_MODELOS = {
    "gpt-4o": {"input": 5.00, "output": 15.00},
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-3.5-turbo": {"input": 0.50, "output": 1.50}
}

# Consumo médio por tipo de query (em tokens)
CONSUMO_MEDIO = {
    "sem_retrieval": {
        "input_tokens": 500,
        "output_tokens": 300,
        "tool_calls": 0,
        "latencia_s": 1.5
    },
    "single_step": {
        "input_tokens": 1500,
        "output_tokens": 500,
        "tool_calls": 1,
        "latencia_s": 4.0
    },
    "multi_step": {
        "input_tokens": 5000,
        "output_tokens": 1000,
        "tool_calls": 3.5,  # média de 3-4 tool calls
        "latencia_s": 18.0
    }
}


def calcular_custo_mensal(
    volume_diario: int = 1000,
    modelo: str = "gpt-4o-mini",
    distribuicao_rotas: dict = None
) -> pd.DataFrame:
    """Calcula o custo mensal do agente RAG.
    
    Args:
        volume_diario: Número de queries por dia
        modelo: Modelo LLM usado
        distribuicao_rotas: % de queries por tipo {sem_retrieval, single_step, multi_step}
    
    Returns:
        DataFrame com análise de custo detalhada
    """
    if distribuicao_rotas is None:
        # Distribuição típica em sistema jurídico
        distribuicao_rotas = {
            "sem_retrieval": 0.30,
            "single_step": 0.50,
            "multi_step": 0.20
        }
    
    precos = PRECOS_MODELOS[modelo]
    resultados = []
    
    for tipo, pct in distribuicao_rotas.items():
        consumo = CONSUMO_MEDIO[tipo]
        queries_dia = volume_diario * pct
        
        # Custo por query (em USD)
        custo_input = (consumo["input_tokens"] / 1_000_000) * precos["input"]
        custo_output = (consumo["output_tokens"] / 1_000_000) * precos["output"]
        custo_query = custo_input + custo_output
        
        # Projeção mensal
        custo_dia = custo_query * queries_dia
        custo_mes = custo_dia * 30
        
        resultados.append({
            "Tipo": tipo,
            "% queries": f"{pct:.0%}",
            "Queries/dia": int(queries_dia),
            "Tokens input": consumo["input_tokens"],
            "Tokens output": consumo["output_tokens"],
            "Tool calls": consumo["tool_calls"],
            "Custo/query (USD)": f"${custo_query:.4f}",
            "Custo/dia (USD)": f"${custo_dia:.2f}",
            "Custo/mês (USD)": f"${custo_mes:.2f}",
            "Latência média": f"{consumo['latencia_s']:.1f}s"
        })
    
    return pd.DataFrame(resultados)


# Análise de custo para 1000 queries/dia
print("💰 ANÁLISE DE CUSTO — AGENTE RAG JURÍDICO")
print("=" * 80)
print(f"Volume: 1.000 queries/dia | Modelo: gpt-4o-mini")
print()

df_custo = calcular_custo_mensal(
    volume_diario=1000,
    modelo="gpt-4o-mini",
    distribuicao_rotas={"sem_retrieval": 0.30, "single_step": 0.50, "multi_step": 0.20}
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
print(df_custo.to_string(index=False))

# Comparativo entre modelos
print("\n📊 COMPARATIVO ENTRE MODELOS (1000 queries/dia, 30 dias):")
print(f"{'Modelo':<20} {'Custo mensal estimado':>25}")
print("-" * 47)

for modelo in PRECOS_MODELOS.keys():
    df = calcular_custo_mensal(1000, modelo)
    # Soma os custos mensais extraindo os valores numéricos
    total = sum(float(v.replace("$", "")) for v in df["Custo/mês (USD)"])
    print(f"  {modelo:<18} ${total:>23.2f}")

---
## Resumo e Próximos Passos

### O que aprendemos nesta aula:

1. **Padrão ReAct** — Ciclo `Thought → Action → Observation` para raciocínio multi-etapa
2. **Agentic RAG** — LLM como orquestrador com múltiplas ferramentas no LangGraph
3. **Design de ferramentas** — Descrições precisas que o LLM entende e usa corretamente
4. **Guards de segurança** — `max_iterations`, timeout e fallbacks para produção
5. **Adaptive RAG** — Classificador de complexidade com 3 caminhos de recuperação
6. **Observabilidade** — LangFuse para rastrear traces, latência e custo
7. **Avaliação** — RAGAS + métricas de trajetória para agentes

### Critérios de Avaliação desta Aula:

- ✅ Agente RAG funcional com 3 ferramentas e trace completo no LangFuse
- ✅ Adaptive RAG com 3 caminhos implementado e distribuição de rotas documentada  
- ✅ Análise de custo: quanto custaria rodar o agente 1.000 vezes?

### Próxima Aula (Aula 11):

**Graph RAG e Raciocínio Estruturado** — Como usar grafos de conhecimento para melhorar a qualidade do retrieval em casos jurídicos complexos.

---

### Referências ABNT

YAO, Shunyu et al. **ReAct: Synergizing Reasoning and Acting in Language Models**. *Proceedings of ICLR 2023*. Disponível em: https://arxiv.org/abs/2210.03629. Acesso em: 2024.

JEONG, Soyeong et al. **Adaptive-RAG: Learning to Adapt Retrieval-Augmented Large Language Models through Question Complexity**. *Findings of ACL 2024*. Disponível em: https://arxiv.org/abs/2403.14403. Acesso em: 2024.

LANGCHAIN-AI. **LangGraph: Build Stateful, Multi-Actor Applications with LLMs**. Disponível em: https://langchain-ai.github.io/langgraph. Acesso em: 2024.